In [0]:

CREATE OR REPLACE TABLE telecom_lakehouse.bronze.network_usage_raw
USING DELTA
COMMENT 'Synthetic raw telecom network usage records'
AS
SELECT
    CONCAT('USG', LPAD(CAST(u.id AS STRING), 10, '0')) AS usage_id,

    CONCAT(
        'SUB',
        LPAD(
            CAST(PMOD(u.id * 37, 100000) + 1 AS STRING),
            8,
            '0'
        )
    ) AS subscription_id,

    DATE_ADD(
        DATE'2025-01-01',
        CAST(PMOD(u.id * 17, 365) AS INT)
    ) AS usage_date,

    CASE
        WHEN PMOD(u.id, 3) = 0 THEN 'VOICE'
        WHEN PMOD(u.id, 3) = 1 THEN 'SMS'
        ELSE 'DATA'
    END AS usage_type,

    CASE
        WHEN PMOD(u.id, 3) = 0 THEN PMOD(u.id * 7, 500) + 1
        WHEN PMOD(u.id, 3) = 1 THEN PMOD(u.id * 11, 100) + 1
        ELSE PMOD(u.id * 13, 5000) + 100
    END AS usage_quantity,

    CASE
        WHEN PMOD(u.id, 3) = 0 THEN 'MINUTES'
        WHEN PMOD(u.id, 3) = 1 THEN 'COUNT'
        ELSE 'MB'
    END AS usage_unit,

    CASE
        WHEN PMOD(u.id, 3) = 0
            THEN ROUND((PMOD(u.id * 7, 500) + 1) * 0.50, 2)
        WHEN PMOD(u.id, 3) = 1
            THEN ROUND((PMOD(u.id * 11, 100) + 1) * 0.10, 2)
        ELSE ROUND((PMOD(u.id * 13, 5000) + 100) * 0.02, 2)
    END AS usage_charge,

    'NETWORK' AS source_system,
    'BATCH_001' AS ingest_batch_id,
    CURRENT_TIMESTAMP() AS ingestion_timestamp

FROM RANGE(1, 2000001) u;